In [6]:
"""
FAKE NEWS DETECTION MODEL - VIVA DEMONSTRATION
Complete script to load and demonstrate your trained model
"""

import joblib
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix



In [7]:
# ==================== LOAD MODEL & COMPONENTS ====================
print("="*70)
print("🔍 FAKE NEWS DETECTION SYSTEM - DEMONSTRATION")
print("="*70)

try:
    model = joblib.load('best_model_part2.pkl')
    vectorizer_tfidf_word = joblib.load('vectorizer_tfidf_word.pkl')
    vectorizer_tfidf_char = joblib.load('vectorizer_tfidf_char.pkl')
    vectorizer_count = joblib.load('vectorizer_count.pkl')
    scaler = joblib.load('scaler.pkl')
    print("✅ All components loaded successfully!")
    print(f"   Model: {type(model).__name__}")
    print(f"   Trees: {model.n_estimators}")
    print(f"   Max Depth: {model.max_depth}")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("\n⚠️  You need to save the vectorizers! Run the training code with:")
    print("   joblib.dump(vectorizer_tfidf_word, 'vectorizer_tfidf_word.pkl')")
    print("   (and similarly for other components)")
    exit()

🔍 FAKE NEWS DETECTION SYSTEM - DEMONSTRATION
✅ All components loaded successfully!
   Model: ExtraTreesClassifier
   Trees: 1200
   Max Depth: 65


In [8]:
# ==================== PREPROCESSING FUNCTIONS ====================
def clean_text_advanced(text):
    """Same cleaning as training"""
    text = str(text).lower()
    
    # Remove fact-check prefixes
    if ':' in text:
        parts = text.split(':', 1)
        if any(word in parts[0] for word in ['fact-check', 'fact check', 'wrong', 'false']):
            text = parts[1]
    
    # Remove leak words
    leak_words = ['fact-check', 'factcheck', 'debunked', 'hoax', 'busted', 'fake', 'false claim']
    for phrase in leak_words:
        text = text.replace(phrase, ' ')
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    
    # Handle multiple punctuation
    text = re.sub(r'!{2,}', ' MULTIEXCLAIM ', text)
    text = re.sub(r'\?{2,}', ' MULTIQUESTION ', text)
    text = re.sub(r'\.{3,}', ' ELLIPSIS ', text)
    
    # Clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_linguistic_features(text):
    """Extract the same 38 features used in training"""
    features = {}
    
    # Split statement and body (assuming [SEP] token)
    if '[SEP]' in text:
        stmt, body = text.split('[SEP]', 1)
    else:
        stmt = text
        body = ""
    
    # Length features
    features['stmt_len'] = len(stmt)
    features['body_len'] = len(body)
    features['combined_len'] = len(text)
    features['stmt_words'] = len(stmt.split())
    features['body_words'] = len(body.split())
    features['total_words'] = features['stmt_words'] + features['body_words']
    
    # Ratio features
    features['stmt_body_ratio'] = features['stmt_len'] / (features['body_len'] + 1)
    features['word_density'] = features['total_words'] / (features['combined_len'] + 1)
    features['avg_word_len'] = features['combined_len'] / (features['total_words'] + 1)
    
    # Punctuation counts
    features['exclaim_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['period_count'] = text.count('.')
    features['comma_count'] = text.count(',')
    features['quote_count'] = text.count('"') + text.count("'")
    features['caps_count'] = sum(1 for c in text if c.isupper())
    features['digit_count'] = sum(1 for c in text if c.isdigit())
    
    # Special markers
    features['has_multiexclaim'] = int('MULTIEXCLAIM' in text)
    features['has_multiquestion'] = int('MULTIQUESTION' in text)
    features['has_ellipsis'] = int('ELLIPSIS' in text)
    
    # Ratios
    features['caps_ratio'] = features['caps_count'] / (features['combined_len'] + 1)
    features['digit_ratio'] = features['digit_count'] / (features['combined_len'] + 1)
    
    # Fake news keywords
    fake_keywords = ['viral', 'shocking', 'breaking', 'exposed', 'revealed',
                     'truth', 'must watch', 'exclusive', 'urgent', 'alert',
                     'proof', 'evidence', 'conspiracy', 'hidden', 'secret']
    for keyword in fake_keywords:
        features[f'has_{keyword}'] = int(keyword in text.lower())
    
    # Sentence features
    features['sentence_count'] = len(re.findall(r'[.!?]', text)) + 1
    features['avg_sentence_len'] = features['total_words'] / features['sentence_count']
    
    return features

def predict_news(statement, body=""):
    """Predict if news is fake or real"""
    # Clean and combine text
    stmt_clean = clean_text_advanced(statement)
    body_clean = clean_text_advanced(body)
    combined = stmt_clean + ' [SEP] ' + body_clean
    
    # Extract linguistic features
    ling_dict = extract_linguistic_features(combined)
    ling_features = np.array([list(ling_dict.values())])
    ling_scaled = scaler.transform(ling_features)
    
    # Vectorize text
    tfidf_word = vectorizer_tfidf_word.transform([combined])
    tfidf_char = vectorizer_tfidf_char.transform([combined])
    count_vec = vectorizer_count.transform([combined])
    
    # Combine all features
    X_combined = hstack([tfidf_word, tfidf_char, count_vec, csr_matrix(ling_scaled)])
    
    # Predict
    prediction = model.predict(X_combined)[0]
    probability = model.predict_proba(X_combined)[0]
    
    return {
        'prediction': 'FAKE' if prediction == 0 else 'TRUE',
        'confidence_fake': probability[0] * 100,
        'confidence_true': probability[1] * 100,
        'linguistic_features': ling_dict
    }

In [9]:
# ==================== DEMONSTRATION EXAMPLES ====================
print("\n" + "="*70)
print("📊 DEMONSTRATION WITH SAMPLE NEWS")
print("="*70)

# Example 1: Likely Fake News
fake_example = {
    'statement': "BREAKING!!! Shocking conspiracy exposed by leaked documents!!!",
    'body': "Viral video reveals hidden truth that authorities don't want you to know. Must watch! This exclusive evidence will shock you. Share immediately before it gets deleted!"
}

print("\n🔴 Example 1 - Suspicious News:")
print(f"Statement: {fake_example['statement']}")
print(f"Body: {fake_example['body'][:100]}...")

result1 = predict_news(fake_example['statement'], fake_example['body'])
print(f"\n📌 Prediction: {result1['prediction']}")
print(f"   Fake Confidence: {result1['confidence_fake']:.2f}%")
print(f"   True Confidence: {result1['confidence_true']:.2f}%")

# Example 2: Likely True News
true_example = {
    'statement': "Government announces new agricultural policy for farmers",
    'body': "The Ministry of Agriculture released details of the revised policy aimed at improving crop insurance coverage and minimum support prices. The policy will be implemented from next quarter."
}

print("\n\n🟢 Example 2 - Credible News:")
print(f"Statement: {true_example['statement']}")
print(f"Body: {true_example['body'][:100]}...")

result2 = predict_news(true_example['statement'], true_example['body'])
print(f"\n📌 Prediction: {result2['prediction']}")
print(f"   Fake Confidence: {result2['confidence_fake']:.2f}%")
print(f"   True Confidence: {result2['confidence_true']:.2f}%")



📊 DEMONSTRATION WITH SAMPLE NEWS

🔴 Example 1 - Suspicious News:
Statement: BREAKING!!! Shocking conspiracy exposed by leaked documents!!!
Body: Viral video reveals hidden truth that authorities don't want you to know. Must watch! This exclusive...


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



📌 Prediction: TRUE
   Fake Confidence: 44.48%
   True Confidence: 55.52%


🟢 Example 2 - Credible News:
Statement: Government announces new agricultural policy for farmers
Body: The Ministry of Agriculture released details of the revised policy aimed at improving crop insurance...


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



📌 Prediction: TRUE
   Fake Confidence: 44.39%
   True Confidence: 55.61%


In [10]:
# ==================== TEST WITH DATABASE (IF PROVIDED) ====================
print("\n\n" + "="*70)
print("📂 TESTING WITH DATABASE SAMPLES")
print("="*70)

try:
    # Load your database
    df = pd.read_excel("bharatfakenewskosh.xlsx")
    print(f"✅ Loaded database: {len(df)} samples")
    
    # Test on 5 random samples
    samples = df.sample(5, random_state=42)
    
    correct = 0
    for idx, row in samples.iterrows():
        statement = row['Eng_Trans_Statement']
        body = row['Eng_Trans_News_Body']
        actual_label = str(row['Label']).upper()  # Convert to uppercase for comparison
        
        result = predict_news(statement, body)
        predicted_label = 'FALSE' if result['prediction'] == 'FAKE' else 'TRUE'
        
        is_correct = predicted_label == actual_label
        correct += is_correct
        
        print(f"\n{'✅' if is_correct else '❌'} Sample {idx}:")
        print(f"   Statement: {statement[:80]}...")
        print(f"   Actual: {actual_label} | Predicted: {predicted_label}")
        print(f"   Confidence: {max(result['confidence_fake'], result['confidence_true']):.1f}%")
    
    accuracy = (correct / 5) * 100
    print(f"\n📊 Accuracy on samples: {accuracy:.1f}% ({correct}/5)")
    
except FileNotFoundError:
    print("⚠️  Database file not found. Place 'bharatfakenewskosh.xlsx' in the same directory.")
except Exception as e:
    print(f"⚠️  Error loading database: {e}")



📂 TESTING WITH DATABASE SAMPLES
✅ Loaded database: 26232 samples


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



❌ Sample 12029:
   Statement: Pakistani students are using Indian flag to escape from Ukraine?...
   Actual: TRUE | Predicted: FALSE
   Confidence: 51.0%


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



✅ Sample 4645:
   Statement: Rafale fuel in the middle sky! Video Share of Brazil claims Vuo...
   Actual: FALSE | Predicted: FALSE
   Confidence: 83.0%


...


✅ Sample 14275:
   Statement: FactCheck: Did the Islamist take part in the farmers' struggle?...
   Actual: FALSE | Predicted: FALSE
   Confidence: 91.1%


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



✅ Sample 16728:
   Statement: Is the 20 -way highway photo in China?...
   Actual: FALSE | Predicted: FALSE
   Confidence: 65.8%


f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



✅ Sample 24133:
   Statement: Rakesh Tikait claims BJP leaders shot at his car, video posted by him shows farm...
   Actual: TRUE | Predicted: TRUE
   Confidence: 84.9%

📊 Accuracy on samples: 80.0% (4/5)


In [12]:
# ==================== INTERACTIVE PREDICTION ====================
print("\n\n" + "="*70)
print("🎯 INTERACTIVE PREDICTION MODE")
print("="*70)
print("Enter your own news to check (or press Enter to skip):\n")

statement_input = input("News Statement: ").strip()
if statement_input:
    body_input = input("News Body (optional): ").strip()
    
    result = predict_news(statement_input, body_input)
    print(f"\n{'🔴' if result['prediction'] == 'FAKE' else '🟢'} Result: {result['prediction']}")
    print(f"Fake Confidence: {result['confidence_fake']:.2f}%")
    print(f"True Confidence: {result['confidence_true']:.2f}%")

print("\n" + "="*70)
print("✅ DEMONSTRATION COMPLETE!")
print("="*70)



🎯 INTERACTIVE PREDICTION MODE
Enter your own news to check (or press Enter to skip):



f:\Fake_News_Detection\Final_Attempt\.finalvenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



🟢 Result: TRUE
Fake Confidence: 37.06%
True Confidence: 62.94%

✅ DEMONSTRATION COMPLETE!
